# Accessing MODFLOW 6 Output

This tutorial shows how to access output from MODFLOW 6 models and packages
by using the built in `.output` attribute on any MODFLOW 6 model or
package object

In [1]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

In [2]:
import numpy as np

In [3]:
# ## Package import
import flopy

## Load a simple demonstration model

In [4]:
exe_name = "mf6"

In [5]:
def get_project_root_path(path=None):
    """
    Infers the path to the project root given the path to the current working directory.
    The current working location must be somewhere in the project, i.e. below the root.

    Parameters
    ----------
    path : the path to the current working directory

    Returns
    -------
        The path to the project root
    """

    cwd = Path(path) if path is not None else Path.cwd()
    if cwd.name == "autotest":
        # we're in top-level autotest folder
        return cwd.parent
    elif "autotest" in cwd.parts and cwd.parts.index(
        "autotest"
    ) > cwd.parts.index("flopy"):
        # we're somewhere inside autotests
        parts = cwd.parts[0 : cwd.parts.index("autotest")]
        return Path(*parts)
    elif "examples" in cwd.parts and cwd.parts.index(
        "examples"
    ) > cwd.parts.index("flopy"):
        # we're somewhere inside examples folder
        parts = cwd.parts[0 : cwd.parts.index("examples")]
        return Path(*parts)
    elif cwd.parts.count("flopy") >= 1:
        # we're somewhere inside the project or flopy module
        tries = [1]
        if "CI" in os.environ:
            tries.append(2)
        for t in tries:
            parts = cwd.parts[0 : cwd.parts.index("flopy") + (t)]
            pth = Path(*parts)
            if (
                next(iter([p for p in pth.glob("pyproject.toml")]), None)
                is not None
            ):
                return pth
        raise Exception(
            f"Can't infer location of project root from {cwd}"
            f"(run from project root, flopy module, examples, or autotest)"
        )
    elif cwd.parts.count("flopy") == 1 and cwd.name == "flopy":
        # we're in project root
        return cwd
    else:
        raise Exception(
            f"Can't infer location of project root from {cwd}"
            f"(run from project root, flopy module, examples, or autotest)"
        )

In [6]:
ws = os.path.abspath(os.path.dirname(""))
sim_ws = str(
    get_project_root_path() / "examples" / "data" / "mf6" / "test001e_UZF_3lay"
)

In [7]:
# load the model
sim = flopy.mf6.MFSimulation.load(
    sim_ws=sim_ws,
    exe_name=exe_name,
    verbosity_level=0,
)
# change the simulation path, rewrite the files, and run the model
temp_dir = TemporaryDirectory()
sim_ws = temp_dir.name
sim.set_sim_path(sim_ws)
sim.write_simulation(silent=True)
sim.run_simulation(silent=True)

(True, [])

## Get output using the `.output` attribute
The output attribute dynamically generates methods for each package based on
the available output options within that package. A list of all available
outputs are:

+-----------------------+------------------------------------------------+
| head()                | Method to get the `HeadFile` object for the    |
|                       | model. Accessed from the model object or the   |
|                       | OC package object                              |
+-----------------------+------------------------------------------------+
| budget()              | Method to get the `CellBudgetFile` object for  |
|                       | the model. Accessed from the model object or   |
|                       | the OC package object                          |
+-----------------------+------------------------------------------------+
| budgetcsv()           | Method to get the MODFLOW-6 budget csv as a    |
|                       | `CsvFile` object. Valid for model, oc, and     |
|                       | advanced packages such as MAW, UZF, LAK        |
+-----------------------+------------------------------------------------+
| zonebudget()          | Method to get the `ZoneBudget6` object for     |
|                       | the model. Accessed from the model object or   |
|                       | the OC package object                          |
+-----------------------+------------------------------------------------+
| obs()                 | Method to get observation file data in the     |
|                       | form of a `MF6Obs` object. Accessed from any   |
|                       | package that allows observations.              |
+-----------------------+------------------------------------------------+
| csv()                 | Method to get csv output data in the form of a |
|                       | `CsvFile` object. Example files are inner and  |
|                       | outer iteration files from IMS                 |
+-----------------------+------------------------------------------------+
| package_convergence() | Method to get csv based package convergence    |
|                       | information from packages such as SFR, LAK,    |
|                       | UZF, and MAW. Returns a `CsvFile` object       |
+-----------------------+------------------------------------------------+
| stage()               | Method to get binary stage file output from    |
|                       | the SFR and LAK packages                       |
+-----------------------+------------------------------------------------+
| concentration()       | Method to get the binary concentration file    |
|                       | output from a groundwater transport model.     |
|                       | Accessed from the model object or the OC       |
|                       | package object                                 |
+-----------------------+------------------------------------------------+
| cim()                 | Method to get immobile concentration output    |
|                       | from the CIM package                           |
+-----------------------+------------------------------------------------+
| density()             | Method to get density file output from the     |
|                       | BUY package                                    |
+-----------------------+------------------------------------------------+

## Get head file and cell budget file outputs
The head file output and cell budget file output can be loaded from either
the model object or the OC package object.

In [8]:
ml = sim.get_model("gwf_1")

In [9]:
bud = ml.output.budget()
bud.get_data(idx=0, full3D=True)

[array([[[ 0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
          -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
           0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
           0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,
          -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.]]])]

In [10]:
budcsv = ml.output.budgetcsv()
budcsv.data

rec.array([(2.00e+00, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   0.        ,   0.        ,  0.00000000e+00),
           (4.00e+00, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   0.        ,   0.        ,  0.00000000e+00),
           (6.00e+00, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   0.        ,   0.        ,  0.00000000e+00),
           (8.00e+00, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   0.        ,   0.        ,  0.00000000e+00),
           (1.00e+01, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   0.        ,   0.        ,  0.00000000e+00),
           (1.20e+01, 0.00000000e+00,  0.        , 0.,   0.        , 0., 0., 0.        ,   0.        ,  0.        , 0., 0., 0.,   

In [11]:
hds = ml.output.head()
hds.get_data()

array([[[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]],

       [[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]],

       [[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]]])

In [12]:
bud = ml.oc.output.budget()
bud.get_data(idx=0, full3D=True)

[array([[[ 0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
          -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
          -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
           0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
           0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,
          -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
           0.,  0., -0., -0.]]])]

In [13]:
hds = ml.oc.output.head()
hds.get_data()

array([[[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]],

       [[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]],

       [[-25.        , -24.93660487, -24.88816332, -24.85548043,
         -24.83902108, -24.83902108, -24.85548043, -24.88816332,
         -24.93660487, -25.        ]]])

## Get output associated with a specific package
The `.output` attribute is tied to the package object and allows the user
to get the output types specified in the MODFLOW 6 package. Here is an
example with a UZF package that has UZF budget file output, budgetcsv
file output, package convergence output, and observation data.

In [14]:
uzf = ml.uzf
uzf_bud = uzf.output.budget()
uzf_bud.get_data(idx=0)

[rec.array([( 1,  9,  0., 100000.), ( 9,  1, -0., 100000.),
            ( 2, 10,  0., 100000.), (10,  2, -0., 100000.),
            ( 3, 11,  0., 100000.), (11,  3, -0., 100000.),
            ( 4, 12,  0., 100000.), (12,  4, -0., 100000.),
            ( 5, 13,  0., 100000.), (13,  5, -0., 100000.),
            ( 6, 14,  0., 100000.), (14,  6, -0., 100000.),
            ( 7, 15,  0., 100000.), (15,  7, -0., 100000.),
            ( 8, 16,  0., 100000.), (16,  8, -0., 100000.),
            ( 9, 17,  0., 100000.), (17,  9, -0., 100000.),
            (10, 18,  0., 100000.), (18, 10, -0., 100000.),
            (11, 19,  0., 100000.), (19, 11, -0., 100000.),
            (12, 20,  0., 100000.), (20, 12, -0., 100000.),
            (13, 21,  0., 100000.), (21, 13, -0., 100000.),
            (14, 22,  0., 100000.), (22, 14, -0., 100000.),
            (15, 23,  0., 100000.), (23, 15, -0., 100000.),
            (16, 24,  0., 100000.), (24, 16, -0., 100000.)],
           dtype=[('node', '<i4'), ('no

In [15]:
uzf_budcsv = uzf.output.budgetcsv()
uzf_budcsv.data

rec.array([(2.00e+00, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.01124365, 7999.98875635, 8000.        , 8000.        ,  3.41060513e-14),
           (4.00e+00, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.02248729, 7999.97751271, 8000.        , 8000.        , -2.27373675e-14),
           (6.00e+00, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.03373091, 7999.96626909, 8000.        , 8000.        ,  6.82121026e-14),
           (8.00e+00, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.04497452, 7999.95502458, 8000.        , 7999.9999991 ,  1.12417183e-08),
           (1.00e+01, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.10119265, 7999.89880735, 8000.        , 8000.        ,  1.13686838e-13),
           (1.20e+01, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0., 0.15741054, 7999.84258946, 8000.        , 8000.        , -1.13686838e-14),
           (1.40e+01, 0., 8000., 0., 0., 0.00000000e+00,   0.        , 0., 0

In [16]:
uzf_conv = uzf.output.package_convergence()
if uzf_conv is not None:
    uzf_conv.data[0:10]

In [17]:
uzf_obs = uzf.output.obs()
uzf_obs.data[0:10]

rec.array([( 2., 0.05, 0.05, 0.), ( 4., 0.05, 0.05, 0.),
           ( 6., 0.05, 0.05, 0.), ( 8., 0.05, 0.05, 0.),
           (10., 0.05, 0.05, 0.), (12., 0.05, 0.05, 0.),
           (14., 0.05, 0.05, 0.), (16., 0.05, 0.05, 0.),
           (18., 0.05, 0.05, 0.), (20., 0.05, 0.05, 0.)],
          dtype=[('totim', '<f8'), ('ID3_DPTH=8.0', '<f8'), ('ID3_DPTH=24.0', '<f8'), ('ID3_RCH', '<f8')])

## Check which output types are available in a package
The `.output` attribute also has a `methods()` function that returns a list
of available output functions for a given package. Here are a couple of
examples

In [18]:
print("UZF package: ", uzf.output.methods())
print("Model object: ", ml.output.methods())
print("OC package: ", ml.oc.output.methods())
print("DIS package: ", ml.dis.output.methods())

UZF package:  ['zonebudget()', 'budget()', 'budgetcsv()', 'package_convergence()', 'obs()']
Model object:  ['list()', 'zonebudget()', 'budget()', 'budgetcsv()', 'head()']
OC package:  ['list()', 'zonebudget()', 'budget()', 'budgetcsv()', 'head()']
DIS package:  None


## Managing multiple observation and csv file outputs in the same package
For many packages, multiple observation output files can be used. The
`obs()` and `csv()` functions allow the user to specify a observation file
or csv file name. If no name is specified, the `obs()` and `csv()` methods
will return the first file that is listed in the package.

In [19]:
output = ml.obs[0].output
obs_names = output.obs_names
output.obs(f=obs_names[0]).data[0:10]

rec.array([( 2., 0.05, 0.05, 0.), ( 4., 0.05, 0.05, 0.),
           ( 6., 0.05, 0.05, 0.), ( 8., 0.05, 0.05, 0.),
           (10., 0.05, 0.05, 0.), (12., 0.05, 0.05, 0.),
           (14., 0.05, 0.05, 0.), (16., 0.05, 0.05, 0.),
           (18., 0.05, 0.05, 0.), (20., 0.05, 0.05, 0.)],
          dtype=[('totim', '<f8'), ('ID3_DPTH=8.0', '<f8'), ('ID3_DPTH=24.0', '<f8'), ('ID3_RCH', '<f8')])

## Creating and running ZoneBudget for MF6
For the model and many packages, zonebudget can be run on the cell budget
file. The `.output` method allows the user to easily build a ZoneBudget6
instance, then run the model, and view output. First we'll build a layered
zone array, then build and run zonebudget

In [20]:
zarr = np.ones(ml.modelgrid.shape, dtype=int)
for i in range(1, 4):
    zarr[i - 1] *= i

In [21]:
zonbud = ml.output.zonebudget(zarr)
zonbud.change_model_ws(sim_ws)
zonbud.write_input()
zonbud.run_model()

FloPy is using the following executable to run the model: ../../home/runner/.local/bin/modflow/zbud6
                              ZONEBUDGET Version 6
                             U.S. GEOLOGICAL SURVEY
                        VERSION 6.4.1 Release 12/09/2022
...........................................................................................
 
Normal Termination


(True, [])

In [22]:
df = zonbud.get_dataframes(net=True)
df = df.reset_index()
df

,totim,name,ZONE_1,ZONE_2,ZONE_3
0,2.0,ZONE_0,0.0,0.000000e+00,0.000000e+00
1,2.0,ZONE_1,0.0,0.000000e+00,0.000000e+00
2,2.0,ZONE_2,0.0,0.000000e+00,0.000000e+00
3,2.0,ZONE_3,0.0,0.000000e+00,0.000000e+00
4,4.0,ZONE_0,0.0,0.000000e+00,0.000000e+00
...,...,...,...,...,...
355,2520.0,ZONE_3,0.0,0.000000e+00,0.000000e+00
356,2560.0,ZONE_0,0.0,0.000000e+00,0.000000e+00
357,2560.0,ZONE_1,0.0,0.000000e+00,0.000000e+00
358,2560.0,ZONE_2,0.0,0.000000e+00,-5.684342e-10


In [23]:
try:
    temp_dir.cleanup()
except:
    # prevent windows permission error
    pass